# Cold-Chain Environmental Monitor Analysis
Upload or mount a run directory containing `readings.csv`, `events.csv`, and `metadata.json`. Outputs are descriptive educational findings, not clinical or compliance decisions.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
RUN_DIR = Path('/content/run')  # change after uploading files
readings = pd.read_csv(RUN_DIR/'readings.csv', parse_dates=['timestamp']).sort_values('timestamp')
events = pd.read_csv(RUN_DIR/'events.csv', parse_dates=['timestamp'])
metadata = json.loads((RUN_DIR/'metadata.json').read_text())
readings.head()


In [ ]:
dt_min = readings['timestamp'].diff().dt.total_seconds().div(60)
readings['rate_c_per_min'] = readings['aht20_temp_c'].diff().div(dt_min.replace(0, np.nan))
readings['temp_disagreement_c'] = (readings['aht20_temp_c']-readings['dht22_temp_c']).abs()
readings['humidity_disagreement_pct'] = (readings['aht20_humidity_pct']-readings['dht22_humidity_pct']).abs()
readings['rolling_mean_c'] = readings['aht20_temp_c'].rolling(6, min_periods=1).mean()
readings['rolling_std_c'] = readings['aht20_temp_c'].rolling(6, min_periods=2).std()
readings.describe(include='all')


In [ ]:
fig, ax = plt.subplots(figsize=(14,5))
ax.plot(readings.timestamp, readings.aht20_temp_c, label='AHT20 primary')
ax.plot(readings.timestamp, readings.dht22_temp_c, label='DHT22 comparison', alpha=.8)
if metadata.get('thresholdEnabled'):
    ax.axhline(metadata['lowerThresholdC'], linestyle='--', label='Lower study threshold')
    ax.axhline(metadata['upperThresholdC'], linestyle='--', label='Upper study threshold')
for _, event in events.iterrows(): ax.axvline(event.timestamp, alpha=.15)
ax.set_ylabel('Temperature (°C)'); ax.set_title('Temperature timeline with recorded events'); ax.legend(); plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(14,5))
ax.plot(readings.timestamp, readings.temp_disagreement_c)
ax.axhline(1.0, linestyle='--', label='Advisory')
ax.axhline(2.0, linestyle='--', label='Fault candidate')
ax.set_ylabel('Absolute difference (°C)'); ax.set_title('AHT20–DHT22 disagreement'); ax.legend(); plt.show()


In [ ]:
summary = {
 'trial_id': metadata.get('trialId'),
 'samples': len(readings),
 'collection_minutes': (readings.timestamp.max()-readings.timestamp.min()).total_seconds()/60,
 'mean_temperature_c': readings.aht20_temp_c.mean(),
 'min_temperature_c': readings.aht20_temp_c.min(),
 'max_temperature_c': readings.aht20_temp_c.max(),
 'max_disagreement_c': readings.temp_disagreement_c.max(),
 'missing_primary_samples': int((~readings.aht_valid.astype(str).str.lower().isin(['true','1'])).sum()),
 'educational_use_only': True
}
summary
